<a href="https://colab.research.google.com/github/minxuan0923/AML_Assignment/blob/mx/Set5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import libraries

In [1]:
import pandas as pd
from scipy.io import arff
import requests
from io import StringIO

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, optimizers, losses
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

## Load Dataset

In [2]:
URL = (
    "https://raw.githubusercontent.com/"
    "minxuan0923/AML_Assignment/main/data/"
    "UCI_Phishing_Training_Dataset.arff"
)

response = requests.get(URL, timeout=30)
response.raise_for_status()

data, meta = arff.loadarff(StringIO(response.text))

df = pd.DataFrame(data)

for col in df.select_dtypes([object]):
    df[col] = pd.to_numeric(df[col].str.decode('utf-8'))

x_data = df.drop(columns =['Result'])
y_data = df['Result']

## Spliting

In [3]:
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, random_state = 42, test_size=0.2, stratify=y_data)


## Encoding

In [4]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

# One-hot-encoding X values
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

# Convert y values - phishing to "1" and legitimate to "0"
y_train = y_train.replace({-1: 1, 1: 0})
y_test = y_test.replace({-1: 1, 1: 0})

## Autoencoder

In [5]:

class Autoencoder(Model):
  def __init__(self, latent_dim, shape):
    super(Autoencoder, self).__init__()
    self.latent_dim = latent_dim
    self.shape = shape
    self.encoder = tf.keras.Sequential([
      layers.Flatten(),
      layers.Dense(latent_dim, activation='relu'),
    ])
    self.decoder = tf.keras.Sequential([
      layers.Dense(int(tf.math.reduce_prod(shape)), activation='sigmoid'),
      layers.Reshape(shape)
    ])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded


shape = X_test_encoded.shape[1:]
latent_dim = 16 # we will experiment with 4, 8, 16, 32
autoencoder = Autoencoder(latent_dim, shape)

autoencoder.compile(optimizer='adam', loss=losses.binary_crossentropy)

history_ae = autoencoder.fit(X_train_encoded, X_train_encoded,
                epochs=10,
                shuffle=True,
                validation_data=(X_test_encoded, X_test_encoded))

X_train_compressed = autoencoder.encoder.predict(X_train_encoded)
X_test_compressed = autoencoder.encoder.predict(X_test_encoded)

Epoch 1/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.4507 - val_loss: 0.3271
Epoch 2/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2670 - val_loss: 0.2193
Epoch 3/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1846 - val_loss: 0.1615
Epoch 4/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1401 - val_loss: 0.1281
Epoch 5/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1135 - val_loss: 0.1073
Epoch 6/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0969 - val_loss: 0.0943
Epoch 7/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0861 - val_loss: 0.0850
Epoch 8/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0786 - val_loss: 0.0787
Epoch 9/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0733 - val_loss: 0.0740
Epoch 10/10
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0695 - val_loss: 0.0710
277/277 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [6]:
print("Before compression:")
print(X_train_encoded.shape)
print(X_test_encoded.shape)

print("After compression:")
print(X_train_compressed.shape)
print(X_test_compressed.shape)

Before compression:
(8844, 68)
(2211, 68)
After compression:
(8844, 16)
(2211, 16)


## Logistic Regression

In [8]:
# Gradient descent
logistic_model = models.Sequential()
logistic_model.add(layers.Normalization(input_shape = [16,], axis = None))
logistic_model.add(layers.Dense(1, activation = 'sigmoid'))

logistic_model.compile(optimizer='adam',
              loss=losses.binary_crossentropy,
              metrics=['accuracy'])

logistic_model.summary()

history_lr = logistic_model.fit(X_train_compressed, y_train, epochs=50,
                    validation_data=(X_test_compressed, y_test))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_1 (Normalization) │ (None, 16)             │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20 (84.00 B)

 Trainable params: 17 (68.00 B)

 Non-trainable params: 3 (16.00 B)

Epoch 1/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4445 - loss: 1.9567 - val_accuracy: 0.4830 - val_loss: 1.3573
Epoch 2/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5590 - loss: 1.0279 - val_accuracy: 0.6422 - val_loss: 0.7461
Epoch 3/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7360 - loss: 0.5730 - val_accuracy: 0.8010 - val_loss: 0.4447
Epoch 4/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8373 - loss: 0.3717 - val_accuracy: 0.8729 - val_loss: 0.3124
Epoch 5/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8846 - loss: 0.2838 - val_accuracy: 0.8982 - val_loss: 0.2518
Epoch 6/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9028 - loss: 0.2414 - val_accuracy: 0.9141 - val_loss: 0.2198
Epoch 7/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9126 - loss: 0.2187 - val_accuracy: 0.9195 - val_loss: 0.2022
Epoch 8/50
277/277 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9179 - loss: 0.2060 - val_accuracy: 0.

In [9]:
test_loss, test_accuracy = logistic_model.evaluate(
    X_test_compressed,
    y_test
)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9276 - loss: 0.1734
Test Accuracy: 0.9276345372200012
Test Loss: 0.17338143289089203
